In [1]:
# -*- coding: utf-8 -*-
"""2D standard-FD benchmark based on Hundsdorfer--Verwer IMEX-CNLF."""

from __future__ import annotations

import gc
import math
import os
from typing import Dict, List, Tuple

import pandas as pd
import torch

from hv_cnlf_common_verified import (
    CaseOptions,
    load_fixed_lhs,
    run_imex_cnlf_case,
    save_case_outputs,
)


# =============================================================================
# Global settings
# =============================================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64
BASE_PATH = os.environ.get("BENCHMARK_BASE_PATH", ".")
OUTPUT_DIR = os.environ.get(
    "BENCHMARK_OUTPUT_DIR",
    os.path.join(BASE_PATH, "verified_fdm_2d_outputs"),
)

X_MIN, X_MAX = -2.0, 2.0
Y_MIN, Y_MAX = -4.0, 4.0
T_FINAL = 1.0
LEFT_BC, RIGHT_BC = -4.0, 2.0

MU_LIST = [1.0e-02,1.0e-03,1.0e-4]
DAE_TARGET_E2 = {1.0e-02:5.98e-03,1.0e-03:1.47e-02,1.0e-4: 2.25e-2}

NUM_SAMPLES = 10000
EVAL_WARMUP = 20
EVAL_REPEAT = 200

# The reference grid contains 101 nodes in t, x, and y. Direct extraction
# therefore requires Nx-1, Ny-1, and Nt-1 to be divisible by 100.
#
# The moving layer is sharp mainly in the x direction.  For each mu, Ny and Nt
# are held fixed while Nx is refined, so the bracket isolates the x-grid
# requirement.  The reported equivalent grid is the first tested case meeting
# the FINAL DAE target; update DAE_TARGET_E2 from the final DAE table first.
CASES: Dict[float, List[Tuple[int, int, int]]] = {
    1.0e-2: [
        (301, 301, 3001),
        (401, 301, 3001),
        (501, 301, 3001),
        (601, 301, 3001),
        (701, 301, 3001),
    ],
    1.0e-3: [
        (401, 501, 8001),
        (601, 501, 8001),
        (801, 501, 8001),
        (1001, 501, 8001),
        (1201, 501, 8001),
    ],
    1.0e-4: [
        (1501, 301, 12001),
        (2001, 301, 12001),
        (2501, 301, 12001),
        (3001, 301, 12001),
        (3501, 301, 12001),
    ],
}

SUMMARY_FILE = os.path.join(OUTPUT_DIR, "2d_HV_IMEX_CNLF_summary.csv")
SELECTED_FILE = os.path.join(OUTPUT_DIR, "2d_HV_IMEX_CNLF_selected.csv")


# =============================================================================
# Problem definition
# =============================================================================

def reference_filename(mu: float) -> str:
    mu_id = int(round(-math.log10(mu)))
    filename = (
        f"2d_U0_all_t_u_x_y_t_mu{mu_id}_101_101_101_Mathematica_620.csv"
    )
    path = os.path.join(BASE_PATH, filename)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Reference file not found: {path}")
    return filename


def phi_minus(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    theta = torch.pi * (y - x) / 4.0
    term = (
        16.0 * torch.pi
        + 2.0 * torch.pi * torch.cos(theta)
        + torch.pi * x * torch.cos(theta)
        - 2.0 * torch.sin(torch.pi * (-4.0 - x + y) / 4.0)
        + 2.0 * torch.sin(torch.pi * (x + y) / 4.0)
    )
    return -torch.sqrt(torch.clamp(term, min=1.0e-12)) / math.sqrt(math.pi)


def phi_plus(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    theta = torch.pi * (y - x) / 4.0
    term = (
        4.0 * torch.pi
        - 2.0 * torch.pi * torch.cos(theta)
        + torch.pi * x * torch.cos(theta)
        - 2.0 * torch.sin(torch.pi * (4.0 - x + y) / 4.0)
        + 2.0 * torch.sin(torch.pi * (x + y) / 4.0)
    )
    return torch.sqrt(torch.clamp(term, min=1.0e-12)) / math.sqrt(math.pi)


def asymptotic_initial_condition(
    x: torch.Tensor,
    y: torch.Tensor,
    mu: float,
) -> torch.Tensor:
    pm_xy = phi_minus(x, y)
    pp_xy = phi_plus(x, y)

    zero_x = torch.zeros_like(y)
    pm_h = phi_minus(zero_x, y)
    pp_h = phi_plus(zero_x, y)
    jump = pp_h - pm_h

    arg_m = torch.clamp(-x * jump / (2.0 * mu), min=-500.0, max=500.0)
    arg_p = torch.clamp(x * jump / (2.0 * mu), min=-500.0, max=500.0)

    u_m = pm_xy + jump / (torch.exp(arg_m) + 1.0)
    u_p = pp_xy - jump / (torch.exp(arg_p) + 1.0)
    return torch.where(x <= 0.0, u_m, u_p)


def source_f(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return (
        torch.cos(torch.pi * x / 4.0)
        * torch.cos(torch.pi * y / 4.0)
    )


# =============================================================================
# Validation and output helpers
# =============================================================================

def validate_case(nx: int, ny: int, nt: int) -> None:
    if min(nx, ny, nt) < 2:
        raise ValueError(
            f"Invalid case (Nx, Ny, Nt)=({nx}, {ny}, {nt}): "
            "all node counts must be at least 2."
        )

    invalid = [
        name
        for name, value in (("Nx", nx), ("Ny", ny), ("Nt", nt))
        if (value - 1) % 100 != 0
    ]
    if invalid:
        raise ValueError(
            f"Invalid case (Nx, Ny, Nt)=({nx}, {ny}, {nt}): "
            + ", ".join(invalid)
            + " minus one must be divisible by 100 for direct LHS extraction."
        )


def save_summary(rows: List[dict]) -> pd.DataFrame:
    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(SUMMARY_FILE, index=False)
    return summary_df


def save_selected_cases(summary_df: pd.DataFrame) -> None:
    if summary_df.empty or "pass_target" not in summary_df.columns:
        pd.DataFrame(columns=summary_df.columns).to_csv(
            SELECTED_FILE, index=False
        )
        return

    passed = summary_df.loc[
        (summary_df["status"] == "success")
        & summary_df["pass_target"].fillna(False).astype(bool)
    ].copy()

    if passed.empty:
        pd.DataFrame(columns=summary_df.columns).to_csv(
            SELECTED_FILE, index=False
        )
        return

    if "spatial_unknowns" not in passed.columns:
        passed["spatial_unknowns"] = (
            (passed["Nx"].astype(int) - 2)
            * (passed["Ny"].astype(int) - 1)
        )

    passed = passed.sort_values(
        ["mu", "spatial_unknowns", "Nt"],
        kind="stable",
    )
    selected = passed.drop_duplicates(subset=["mu"], keep="first")
    selected.to_csv(SELECTED_FILE, index=False)


# =============================================================================
# Main benchmark
# =============================================================================

def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("=" * 104)
    print("2D Hundsdorfer--Verwer CD2--IMEX-CNLF benchmark")
    print(f"device={DEVICE}, dtype={DTYPE}")
    print("x: Dirichlet; y: periodic; FFT fast Helmholtz solver")
    print("T_total = T_solve + T_eval; setup/error/CPU transfer/output excluded")
    print("Fixed DAE LHS set; direct grid-node extraction; no interpolation")
    print("=" * 104)

    if DEVICE.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")

    rows: List[dict] = []

    for mu in MU_LIST:
        reference = reference_filename(mu)
        lhs = load_fixed_lhs(
            reference_path=os.path.join(BASE_PATH, reference),
            coordinate_columns=("t", "x", "y"),
            index_path=os.path.join(
                BASE_PATH,
                f"2d_LHS_sample_indices_mu{mu:.0e}.npy",
            ),
            num_samples=NUM_SAMPLES,
            save_points_path=os.path.join(
                OUTPUT_DIR,
                f"2d_HV_LHS_points_mu{mu:.0e}.csv",
            ),
        )

        for nx, ny, nt in CASES[mu]:
            validate_case(nx, ny, nt)

            hx = (X_MAX - X_MIN) / (nx - 1)
            hy = (Y_MAX - Y_MIN) / (ny - 1)
            dt = T_FINAL / (nt - 1)

            print("-" * 104)
            print(
                f"mu={mu:.0e}, grid={nx}x{ny}, Nt={nt}, "
                f"hx={hx:.6e}, hy={hy:.6e}, dt={dt:.6e}"
            )

            options = CaseOptions(
                dim=2,
                mu=mu,
                grid=(nx, ny),
                nt=nt,
                bounds=((X_MIN, X_MAX), (Y_MIN, Y_MAX)),
                t_final=T_FINAL,
                left_bc=LEFT_BC,
                right_bc=RIGHT_BC,
                dtype=DTYPE,
                device=DEVICE,
                eval_warmup=EVAL_WARMUP,
                eval_repeat=EVAL_REPEAT,
                finite_check_interval=max(1, (nt - 1) // 100),
                progress_every=max(1, (nt - 1) // 20),
                cfl_warning=1.0,
            )

            try:
                result = run_imex_cnlf_case(
                    options=options,
                    lhs_data=lhs,
                    source_function=source_f,
                    initial_function=asymptotic_initial_condition,
                )

                result["target_e2"] = float(DAE_TARGET_E2[mu])
                result["pass_target"] = (
                    float(result["e2"]) <= float(DAE_TARGET_E2[mu])
                )

                prefix = os.path.join(
                    OUTPUT_DIR,
                    (
                        f"2d_HV_IMEX_CNLF_mu{mu:.0e}"
                        f"_Nx{nx}_Ny{ny}_Nt{nt}"
                    ),
                )
                pred_path, runtime_path = save_case_outputs(
                    result,
                    lhs,
                    ("t", "x", "y"),
                    prefix,
                )

                row = {
                    key: value
                    for key, value in result.items()
                    if key != "prediction"
                }
                row.update(
                    {
                        "status": "success",
                        "failure_reason": "",
                        "Nx": nx,
                        "Ny": ny,
                        "Nt": nt,
                        "hx": hx,
                        "hy": hy,
                        "dt": dt,
                        "prediction_csv": pred_path,
                        "runtime_json": runtime_path,
                    }
                )

                print(
                    f"e2={result['e2']:.6e}, "
                    f"einf={result['einf']:.6e}, "
                    f"target={DAE_TARGET_E2[mu]:.6e}, "
                    f"pass={result['pass_target']}, "
                    f"T_solve={result['T_solve']:.6f}s, "
                    f"T_eval={result['T_eval']:.6e}s, "
                    f"T_total={result['T_total']:.6f}s, "
                    "peak GPU allocated="
                    f"{result['peak_gpu_allocated_gib']:.3f} GiB"
                )

            # Do not use torch.OutOfMemoryError: it does not exist in some
            # PyTorch versions. CUDA OOM is already covered by RuntimeError.
            except (RuntimeError, MemoryError) as exc:
                error_message = f"{type(exc).__name__}: {exc}"
                row = {
                    "status": "failed",
                    "failure_reason": error_message,
                    "mu": mu,
                    "Nx": nx,
                    "Ny": ny,
                    "Nt": nt,
                    "hx": hx,
                    "hy": hy,
                    "dt": dt,
                    "target_e2": float(DAE_TARGET_E2[mu]),
                    "pass_target": False,
                }
                print(f"FAILED: {error_message}")

            rows.append(row)
            save_summary(rows)

            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    summary_df = save_summary(rows)
    save_selected_cases(summary_df)

    print("=" * 104)
    print(f"Saved summary: {SUMMARY_FILE}")
    print(f"Saved selected passing cases: {SELECTED_FILE}")


if __name__ == "__main__":
    main()


2D Hundsdorfer--Verwer CD2--IMEX-CNLF benchmark
device=cuda, dtype=torch.float64
x: Dirichlet; y: periodic; FFT fast Helmholtz solver
T_total = T_solve + T_eval; setup/error/CPU transfer/output excluded
Fixed DAE LHS set; direct grid-node extraction; no interpolation
GPU: NVIDIA H200
Loaded fixed DAE LHS set: N_test=10000, reference=2d_U0_all_t_u_x_y_t_mu2_101_101_101_Mathematica_620.csv, indices=2d_LHS_sample_indices_mu1e-02.npy
--------------------------------------------------------------------------------------------------------
mu=1e-02, grid=301x301, Nt=3001, hx=1.333333e-02, hy=2.666667e-02, dt=3.333333e-04
step=1/3000, t=3.333333e-04, max|u|=4.283409e+00
step=150/3000, t=5.000000e-02, max|u|=4.883291e+00
step=300/3000, t=1.000000e-01, max|u|=4.909410e+00
step=450/3000, t=1.500000e-01, max|u|=4.927390e+00
step=600/3000, t=2.000000e-01, max|u|=4.947418e+00
step=750/3000, t=2.500000e-01, max|u|=4.965127e+00
step=900/3000, t=3.000000e-01, max|u|=4.983182e+00
step=1050/3000, t=3.500